In [7]:
#1. Importar librerías
import os
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import text
from dotenv import load_dotenv

# Cargar variables de entorno ocultas
load_dotenv()

True

In [8]:
#2. Leer CSV desde la carpeta raw
df = pd.read_csv("../01.data/raw/telco_churn.csv")
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [10]:
#3. Limpieza Profesional de Datos - Eliminar filas vacías en la columna "TotalCharges" y convertir a numérico
df_clean = df.copy()

# Forzar conversión de la columna a numérica (los espacios vacíos pasan a ser NaN)
df_clean["TotalCharges"] = pd.to_numeric(df_clean["TotalCharges"], errors="coerce")

# Registrar cuántas filas vacías se van a eliminar para auditoría de datos
filas_nulas = df_clean["TotalCharges"].isna().sum()
print(f"Filas eliminadas por valores faltantes en TotalCharges: {filas_nulas}")

# Eliminar registros vacíos y guardar
df_clean = df_clean.dropna()
df_clean.to_csv("../01.data/processed/telco_churn_clean.csv", index=False)
print("✅ Dataset limpio guardado correctamente en la carpeta 'processed'.")

Filas eliminadas por valores faltantes en TotalCharges: 11
✅ Dataset limpio guardado correctamente en la carpeta 'processed'.


In [11]:
#4. Ingesta segura en PostgreSQL (Estructura moderna de SQLAlchemy)
# URL de conexión segura al contenedor Docker de PostgreSQL
DATABASE_URL = os.getenv("DATABASE_URL", "postgresql://ds_user:ds_password@db:5432/customer_churn")
engine = sa.create_engine(DATABASE_URL)

# Clonamos y mapeamos exactamente las columnas al estándar snake_case de Postgres
df_to_sql = df_clean.copy()
df_to_sql = df_to_sql.rename(columns={
    'customerID': 'customer_id', 'Gender': 'gender', 'SeniorCitizen': 'senior_citizen', 
    'Partner': 'partner', 'Dependents': 'dependents', 'tenure': 'tenure', 
    'PhoneService': 'phone_service', 'MultipleLines': 'multiple_lines', 
    'InternetService': 'internet_service', 'OnlineSecurity': 'online_security', 
    'OnlineBackup': 'online_backup', 'DeviceProtection': 'device_protection', 
    'TechSupport': 'tech_support', 'StreamingTV': 'streaming_tv', 
    'StreamingMovies': 'streaming_movies', 'Contract': 'contract', 
    'PaperlessBilling': 'paperless_billing', 'PaymentMethod': 'payment_method', 
    'MonthlyCharges': 'monthly_charges', 'TotalCharges': 'total_charges', 
    'Churn': 'churn'
})

try:
    with engine.connect() as conn:
        print("🧹 Eliminando tablas antiguas corruptas en Docker...")
        # Forzamos la eliminación de la tabla vieja generada por Pandas para liberar el espacio
        conn.execute(sa.text("DROP TABLE IF EXISTS customers CASCADE;"))
        conn.commit()
        
        print("🏗️ Creando nueva estructura de base de datos relacional y vistas...")
        # Leemos y aplicamos las restricciones e integridad del archivo SQL
        with open("../02.sql/create_tables.sql", "r") as f:
            conn.execute(sa.text(f.read()))
        conn.commit()
            
        print("📥 Insertando registros optimizados de clientes...")
        # Insertamos usando 'append' para respetar las llaves primarias del script SQL
        df_to_sql.to_sql("customers", con=conn, if_exists="append", index=False)
        conn.commit()
        
    print("✅ ¡ÉXITO ROTUNDO! La base de datos se ha reestructurado e ingestó correctamente.")
except Exception as e:
    print(f"❌ Error crítico en la base de datos: {e}")


🧹 Eliminando tablas antiguas corruptas en Docker...
🏗️ Creando nueva estructura de base de datos relacional y vistas...
📥 Insertando registros optimizados de clientes...
✅ ¡ÉXITO ROTUNDO! La base de datos se ha reestructurado e ingestó correctamente.


In [12]:
#5. Creación del dataset de Scoring
# Tomar una muestra aleatoria de 1000 clientes para simular datos nuevos
df_nuevos = df_clean.sample(1000, random_state=42)

# Guardar en la ruta correspondiente
df_nuevos.to_csv("../01.data/scoring/clientes_nuevos.csv", index=False)
print(f"✅ clientes_nuevos.csv creado con {len(df_nuevos)} registros para la fase de scoring.")


✅ clientes_nuevos.csv creado con 1000 registros para la fase de scoring.
